# ML Project / NASDAQ Prediction and Analysis

**Objective** : Predicting if the Nasdaq-100 (`^NDX`) goes up or down **during the New York session of day `t`** (from the open at 09:30 ET to the close at 16:00 ET), using only information known **before 09:00 ET of day `t`**. The main new information is how the NQ futures traded overnight (Asia and Europe); the daily data (stocks, macro) is used as context from the day before.


In [89]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter 
from sklearn.model_selection import TimeSeriesSplit

In [90]:
path_to_data = ("./data")

## Data

We take the current Nasdaq-100 list from Wikipedia. This means we use today's companies for the whole period, so there's some survivorship bias (we'll mention it in the report).

In [91]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [92]:

NQ = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tgtanalytics/nq-futures-1min-bar-2022-2025",
  "Dataset_NQ_1min_2022_2025.csv"
)

In [93]:
TICKERS

['ADBE',
 'AMD',
 'ABNB',
 'ALNY',
 'GOOGL',
 'GOOG',
 'AMZN',
 'AEP',
 'AMGN',
 'ADI',
 'AAPL',
 'AMAT',
 'APP',
 'ARM',
 'ASML',
 'ALAB',
 'ADSK',
 'ADP',
 'AXON',
 'BKR',
 'BKNG',
 'AVGO',
 'CDNS',
 'CTAS',
 'CSCO',
 'CCEP',
 'CMCSA',
 'CEG',
 'CPRT',
 'CRWV',
 'COST',
 'CRWD',
 'CSX',
 'DDOG',
 'DXCM',
 'FANG',
 'DASH',
 'EXC',
 'FAST',
 'FER',
 'FTNT',
 'GEHC',
 'GILD',
 'HONA',
 'HON',
 'IDXX',
 'INTC',
 'INTU',
 'ISRG',
 'KDP',
 'KLAC',
 'LRCX',
 'LIN',
 'LITE',
 'MAR',
 'MRVL',
 'MELI',
 'META',
 'MCHP',
 'MU',
 'MSFT',
 'MSTR',
 'MRNA',
 'MDLZ',
 'MPWR',
 'MNST',
 'NBIS',
 'NFLX',
 'NVDA',
 'NXPI',
 'ORLY',
 'ODFL',
 'PCAR',
 'PLTR',
 'PANW',
 'PAYX',
 'PYPL',
 'PDD',
 'PEP',
 'QCOM',
 'REGN',
 'RKLB',
 'ROP',
 'ROST',
 'SNDK',
 'STX',
 'SHOP',
 'SPCX',
 'SBUX',
 'SNPS',
 'TMUS',
 'TTWO',
 'TER',
 'TSLA',
 'TXN',
 'TRI',
 'VRTX',
 'WMT',
 'WDC',
 'WDAY',
 'XEL']

On top of the stocks we add some macro data: volatility indices (VIX, VXN), treasury yields, oil, gold, silver, the dollar and Bitcoin. The idea is to give the model some context about the market, not just prices.

In [94]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---                    
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    "BTC-USD": "Bitcoin",                     # Bitcoin USD (Risk-on sentiment proxy)
}
tickers1 = list(NASDAQ_FEATURES.keys())

In [95]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1))

In [96]:
df_nasdaq = yf.download("^NDX", start="2022-10-01", end="2025-12-12",
                     auto_adjust=True, progress=False)[['Open', 'Close','Volume']]
print(df_nasdaq.columns)
print(df_nasdaq.tail())

MultiIndex([(  'Open', '^NDX'),
            ( 'Close', '^NDX'),
            ('Volume', '^NDX')],
           names=['Price', 'Ticker'])
Price               Open         Close      Volume
Ticker              ^NDX          ^NDX        ^NDX
Date                                              
2025-12-05  25657.779297  25692.050781  8248280000
2025-12-08  25758.820312  25627.949219  8533670000
2025-12-09  25593.669922  25668.689453  7192600000
2025-12-10  25631.589844  25776.439453  7987900000
2025-12-11  25598.390625  25686.689453  8337770000


In [97]:
df_nasdaq.head(20)

Price,Open,Close,Volume
Ticker,^NDX,^NDX,^NDX
Date,,,
2022-10-03,11059.169922,11229.730469,4415440000
2022-10-04,11477.849609,11582.540039,5004780000
2022-10-05,11429.469727,11573.179688,4091860000
2022-10-06,11556.400391,11485.500000,4423320000
2022-10-07,11272.459961,11039.469727,4634190000
2022-10-10,11048.509766,10926.969727,3989640000
2022-10-11,10865.320312,10791.349609,4738840000
2022-10-12,10810.299805,10785.620117,4091030000


In [98]:
raw = yf.download(TICKERS, start="2022-10-01", end="2025-12-12")[['Open', 'Close','Volume']]

[**********************45%                       ]  51 of 113 completed$HONA: Data doesn't exist for startDate = 1664596800, endDate = 1765515600
[**********************95%*********************  ]  107 of 113 completed$SPCX: Data doesn't exist for startDate = 1664596800, endDate = 1765515600
[*********************100%***********************]  113 of 113 completed

2 Failed downloads:
['HONA', 'SPCX']: Data doesn't exist for startDate = 1664596800, endDate = 1765515600


In [99]:
raw.head(10)

Price             Open                                                  \
Ticker            AAPL        ABNB        ADBE         ADI         ADP   
Date                                                                     
2022-10-01         NaN         NaN         NaN         NaN         NaN   
2022-10-02         NaN         NaN         NaN         NaN         NaN   
2022-10-03  135.458495  105.489998  278.269989  132.337472  208.047462   
2022-10-04  142.142746  109.139999  290.950012  138.866322  214.078875   
2022-10-05  141.201842  110.519997  291.410004  139.662528  214.497956   
2022-10-06  142.907199  112.000000  297.149994  141.751399  218.096797   
2022-10-07  139.702309  108.870003  291.359985  138.323005  211.327340   
2022-10-08         NaN         NaN         NaN         NaN         NaN   
2022-10-09         NaN         NaN         NaN         NaN         NaN   
2022-10-10  137.624512  108.550003  289.750000  135.953197  207.601002   

Price                                                          ...  \
Ticker            ADSK        AEP ALAB        ALNY       AMAT  ...   
Date                                                           ...   
2022-10-01         NaN        NaN  NaN         NaN        NaN  ...   
2022-10-02         NaN        NaN  NaN         NaN        NaN  ...   
2022-10-03  187.830002  75.203011  NaN  205.100006  80.348078  ...   
2022-10-04  197.669998  75.949056  NaN  199.270004  86.230982  ...   
2022-10-05  196.139999  76.720802  NaN  199.809998  84.808639  ...   
2022-10-06  203.809998  75.014376  NaN  199.899994  86.095511  ...   
2022-10-07  201.589996  72.501886  NaN  201.000000  82.409044  ...   
2022-10-08         NaN        NaN  NaN         NaN        NaN  ...   
2022-10-09         NaN        NaN  NaN         NaN        NaN  ...   
2022-10-10  194.660004  71.387106  NaN  194.460007  79.341776  ...   

Price           Volume                                                      \
Ticker             WDC         WMT        XEL ^FVX ^IRX         ^IXIC ^TNX   
Date                                                                         
2022-10-01         NaN         NaN        NaN  NaN  NaN           NaN  NaN   
2022-10-02         NaN         NaN        NaN  NaN  NaN           NaN  NaN   
2022-10-03   5490185.0  17574900.0  2931500.0  0.0  0.0  4.415440e+09  0.0   
2022-10-04  13472374.0  16134300.0  2754400.0  0.0  0.0  5.004780e+09  0.0   
2022-10-05   9325827.0  14553900.0  3003000.0  0.0  0.0  4.091860e+09  0.0   
2022-10-06  11980823.0  14200500.0  3788300.0  0.0  0.0  4.423320e+09  0.0   
2022-10-07  12858766.0  17080500.0  6476400.0  0.0  0.0  4.634190e+09  0.0   
2022-10-08         NaN         NaN        NaN  NaN  NaN           NaN  NaN   
2022-10-09         NaN         NaN        NaN  NaN  NaN           NaN  NaN   
2022-10-10   4845884.0  11411700.0  3970400.0  0.0  0.0  3.989640e+09  0.0   

Price                      
Ticker     ^TYX ^VIX ^VXN  
Date                       
2022-10-01  NaN  NaN  NaN  
2022-10-02  NaN  NaN  NaN  
2022-10-03  0.0  0.0  0.0  
2022-10-04  0.0  0.0  0.0  
2022-10-05  0.0  0.0  0.0  
2022-10-06  0.0  0.0  0.0  
2022-10-07  0.0  0.0  0.0  
2022-10-08  NaN  NaN  NaN  
2022-10-09  NaN  NaN  NaN  
2022-10-10  0.0  0.0  0.0  

[10 rows x 339 columns]

Bitcoin trades 24/7, so before the join we keep its close on its own calendar (weekends included). We use it later for the Bitcoin features.

In [100]:
# BTC close on its own daily calendar, weekends included
btc_close = raw[("Close", "BTC-USD")].dropna()

We join everything on the Nasdaq-100 dates (inner join). Bitcoin trades on weekends but the index doesn't, so this way we only keep real trading days and we don't need to fill anything.

In [101]:
raw = pd.concat((df_nasdaq,raw),axis=1,join='inner',)

In [102]:
raw.loc[:, (slice(None), 'BTC-USD')]

Price,Open,Close,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD
Date,,,
2022-10-03,19044.068359,19623.580078,30484729489
2022-10-04,19623.583984,20336.843750,35887278685
2022-10-05,20335.900391,20160.716797,33223790572
2022-10-06,20161.039062,19955.443359,34711412966
2022-10-07,19957.558594,19546.849609,29227315390
...,...,...,...
2025-12-05,92133.648438,89387.757812,63256398633
2025-12-08,90424.585938,90640.203125,57394099056


In [103]:
raw

Price               Open         Close      Volume        Open              \
Ticker              ^NDX          ^NDX        ^NDX        AAPL        ABNB   
Date                                                                         
2022-10-03  11059.169922  11229.730469  4415440000  135.458495  105.489998   
2022-10-04  11477.849609  11582.540039  5004780000  142.142746  109.139999   
2022-10-05  11429.469727  11573.179688  4091860000  141.201842  110.519997   
2022-10-06  11556.400391  11485.500000  4423320000  142.907199  112.000000   
2022-10-07  11272.459961  11039.469727  4634190000  139.702309  108.870003   
...                  ...           ...         ...         ...         ...   
2025-12-05  25657.779297  25692.050781  8248280000  279.566033  120.919998   
2025-12-08  25758.820312  25627.949219  8533670000  277.164420  124.214996   
2025-12-09  25593.669922  25668.689453  7192600000  277.194321  121.610001   
2025-12-10  25631.589844  25776.439453  7987900000  276.785711  125.449997   
2025-12-11  25598.390625  25686.689453  8337770000  278.131051  127.599998   

Price                                                                   ...  \
Ticker            ADBE         ADI         ADP        ADSK         AEP  ...   
Date                                                                    ...   
2022-10-03  278.269989  132.337472  208.047462  187.830002   75.203011  ...   
2022-10-04  290.950012  138.866322  214.078875  197.669998   75.949056  ...   
2022-10-05  291.410004  139.662528  214.497956  196.139999   76.720802  ...   
2022-10-06  297.149994  141.751399  218.096797  203.809998   75.014376  ...   
2022-10-07  291.359985  138.323005  211.327340  201.589996   72.501886  ...   
...                ...         ...         ...         ...         ...  ...   
2025-12-05  330.600006  276.659838  252.083800  306.000000  114.353964  ...   
2025-12-08  347.450012  279.282235  253.502656  307.600006  114.033799  ...   
2025-12-09  341.399994  275.119470  251.782584  301.070007  112.811378  ...   
2025-12-10  342.130005  273.206590  250.101377  299.459991  113.180056  ...   
2025-12-11  341.649994  279.074086  252.268475  299.929993  111.094189  ...   

Price           Volume                                                      \
Ticker             WDC         WMT        XEL ^FVX ^IRX         ^IXIC ^TNX   
Date                                                                         
2022-10-03   5490185.0  17574900.0  2931500.0  0.0  0.0  4.415440e+09  0.0   
2022-10-04  13472374.0  16134300.0  2754400.0  0.0  0.0  5.004780e+09  0.0   
2022-10-05   9325827.0  14553900.0  3003000.0  0.0  0.0  4.091860e+09  0.0   
2022-10-06  11980823.0  14200500.0  3788300.0  0.0  0.0  4.423320e+09  0.0   
2022-10-07  12858766.0  17080500.0  6476400.0  0.0  0.0  4.634190e+09  0.0   
...                ...         ...        ...  ...  ...           ...  ...   
2025-12-05   5815300.0  23956400.0  3563000.0  0.0  0.0  8.248280e+09  0.0   
2025-12-08   4272200.0  22700700.0  3197700.0  0.0  0.0  8.533670e+09  0.0   
2025-12-09   3709900.0  24326700.0  4842400.0  0.0  0.0  7.192600e+09  0.0   
2025-12-10   7966000.0  22859600.0  4824700.0  0.0  0.0  7.987900e+09  0.0   
2025-12-11   7468900.0  21586600.0  5494900.0  0.0  0.0  8.337770e+09  0.0   

Price                      
Ticker     ^TYX ^VIX ^VXN  
Date                       
2022-10-03  0.0  0.0  0.0  
2022-10-04  0.0  0.0  0.0  
2022-10-05  0.0  0.0  0.0  
2022-10-06  0.0  0.0  0.0  
2022-10-07  0.0  0.0  0.0  
...         ...  ...  ...  
2025-12-05  0.0  0.0  0.0  
2025-12-08  0.0  0.0  0.0  
2025-12-09  0.0  0.0  0.0  
2025-12-10  0.0  0.0  0.0  
2025-12-11  0.0  0.0  0.0  

[802 rows x 342 columns]

### Missing values

Some companies are very recent (ARM, CRWV, SNDK...), so they have almost no history. We drop the columns with less than 40% of data. We picked this threshold by looking at which columns were left.

We also drop GOOG (almost the same as GOOGL) and ^IXIC (it moves with the Nasdaq-100), they don't add anything new.

In [104]:
dropping = raw.isnull().sum().sort_values(ascending=False)/len(raw)

In [105]:
dropping

Price   Ticker
Close   HONA      1.0
Open    SPCX      1.0
Volume  HONA      1.0
Open    HONA      1.0
Close   SPCX      1.0
                 ... 
        ABNB      0.0
        AAPL      0.0
Open    ^VXN      0.0
        ^VIX      0.0
Volume  ^VXN      0.0
Length: 342, dtype: float64

In [106]:
raw_drop = raw.dropna(axis=1, thresh=int(0.4 * len(raw)))
# GOOG is basically GOOGL and ^IXIC moves with ^NDX, so they add nothing
raw_drop = raw_drop.drop(columns=["GOOG", "^IXIC"], level=1)

In [107]:
print(list(set(raw.columns.tolist()) - set(raw_drop.columns.tolist())))

[('Volume', 'NBIS'), ('Close', 'SNDK'), ('Volume', 'CRWV'), ('Volume', 'GOOG'), ('Open', 'NBIS'), ('Open', 'GOOG'), ('Open', 'CRWV'), ('Volume', '^IXIC'), ('Volume', 'HONA'), ('Close', 'NBIS'), ('Volume', 'SPCX'), ('Close', 'CRWV'), ('Open', 'HONA'), ('Close', 'GOOG'), ('Open', '^IXIC'), ('Open', 'SPCX'), ('Volume', 'SNDK'), ('Close', '^IXIC'), ('Close', 'HONA'), ('Open', 'SNDK'), ('Close', 'SPCX')]


In [108]:
try:
    raw_drop.loc[:, (slice(None), "SPCX")]
except KeyError:
    print("SPCX not in raw_drop: it was dropped")

SPCX not in raw_drop: it was dropped


In [109]:
raw.loc[:, (slice(None), "SPCX")]

Price,Open,Close,Volume
Ticker,SPCX,SPCX,SPCX
Date,,,
2022-10-03,NaN,NaN,NaN
2022-10-04,NaN,NaN,NaN
2022-10-05,NaN,NaN,NaN
2022-10-06,NaN,NaN,NaN
2022-10-07,NaN,NaN,NaN
...,...,...,...
2025-12-05,NaN,NaN,NaN
2025-12-08,NaN,NaN,NaN


In [110]:
raw_drop.describe()

Price           Open         Close        Volume        Open              \
Ticker          ^NDX          ^NDX          ^NDX        AAPL        ABNB   
count     802.000000    802.000000  8.020000e+02  802.000000  802.000000   
mean    17930.577398  17937.509013  6.394485e+09  195.785234  129.156884   
std      4099.296659   4093.282944  2.096899e+09   35.868052   16.437167   
min     10481.580078  10679.339844  2.184080e+09  123.706265   82.970001   
25%     14860.862305  14880.222412  4.839432e+09  170.467165  118.917503   
50%     18166.440430  18222.794922  5.532135e+09  191.142670  128.820000   
75%     21079.637695  21090.250000  7.958890e+09  223.816258  139.589996   
max     26147.720703  26119.849609  1.630873e+10  285.206384  169.690002   

Price                                                               ...  \
Ticker        ADBE         ADI         ADP        ADSK         AEP  ...   
count   802.000000  802.000000  802.000000  802.000000  802.000000  ...   
mean    446.934389  195.847190  245.912941  247.925786   86.247909  ...   
std      93.084899   29.608776   33.947381   41.678082   13.414737  ...   
min     278.269989  126.174009  187.019701  183.470001   62.800600  ...   
25%     359.055000  173.815888  219.594199  208.622498   75.419285  ...   
50%     444.264999  190.776042  234.756946  245.160004   82.185062  ...   
75%     524.274979  221.426108  281.185543  290.440002   96.891117  ...   
max     633.030029  279.282235  316.343037  326.880005  120.708565  ...   

Price         Volume                                                          \
Ticker          WDAY           WDC           WMT           XEL   ^FVX   ^IRX   
count   8.020000e+02  8.020000e+02  8.020000e+02  8.020000e+02  802.0  802.0   
mean    2.315561e+06  7.233561e+06  1.824552e+07  3.862529e+06    0.0    0.0   
std     2.162678e+06  4.040634e+06  8.961242e+06  2.032454e+06    0.0    0.0   
min     6.764000e+05  1.615912e+06  6.287500e+06  9.439000e+05    0.0    0.0   
25%     1.452950e+06  4.699064e+06  1.321095e+07  2.695950e+06    0.0    0.0   
50%     1.888150e+06  6.041282e+06  1.572990e+07  3.400650e+06    0.0    0.0   
75%     2.555525e+06  8.560006e+06  2.002980e+07  4.324425e+06    0.0    0.0   
max     4.687410e+07  3.557269e+07  9.765870e+07  2.278220e+07    0.0    0.0   

Price                               
Ticker   ^TNX   ^TYX   ^VIX   ^VXN  
count   802.0  802.0  802.0  802.0  
mean      0.0    0.0    0.0    0.0  
std       0.0    0.0    0.0    0.0  
min       0.0    0.0    0.0    0.0  
25%       0.0    0.0    0.0    0.0  
50%       0.0    0.0    0.0    0.0  
75%       0.0    0.0    0.0    0.0  
max       0.0    0.0    0.0    0.0  

[8 rows x 321 columns]

In [111]:
from fastai.tabular.all import add_datepart

raw_drop = raw_drop.reset_index()
raw_time = add_datepart(raw_drop, 'Date', drop=False)

ModuleNotFoundError: No module named 'fastai'

From the date we keep only `Dayofweek`. The other date columns (year, month, elapsed...) are trends or noise for this target.

In [ ]:
date_cols = ["Year", "Month", "Week", "Day", "Dayofyear", "Is_month_end", "Is_month_start",
             "Is_quarter_end", "Is_quarter_start", "Is_year_end", "Is_year_start", "Elapsed"]
raw_time = raw_time.drop(columns=date_cols, level=0)

In [ ]:
raw_time.isnull().sum().sort_values(ascending=False)/len(raw)

Price   Ticker
Close   ALAB      0.457606
Volume  ALAB      0.457606
Open    ALAB      0.457606
Close   ARM       0.296758
Open    ARM       0.296758
                    ...   
        AMD       0.000000
        AMGN      0.000000
        AMZN      0.000000
        APP       0.000000
        ^NDX      0.000000
Length: 323, dtype: float64

In [ ]:
columns = raw_time.columns.to_list()
print(columns)

[('Date', ''), ('Open', '^NDX'), ('Close', '^NDX'), ('Volume', '^NDX'), ('Open', 'AAPL'), ('Open', 'ABNB'), ('Open', 'ADBE'), ('Open', 'ADI'), ('Open', 'ADP'), ('Open', 'ADSK'), ('Open', 'AEP'), ('Open', 'ALAB'), ('Open', 'ALNY'), ('Open', 'AMAT'), ('Open', 'AMD'), ('Open', 'AMGN'), ('Open', 'AMZN'), ('Open', 'APP'), ('Open', 'ARM'), ('Open', 'ASML'), ('Open', 'AVGO'), ('Open', 'AXON'), ('Open', 'BKNG'), ('Open', 'BKR'), ('Open', 'BTC-USD'), ('Open', 'CCEP'), ('Open', 'CDNS'), ('Open', 'CEG'), ('Open', 'CL=F'), ('Open', 'CMCSA'), ('Open', 'COST'), ('Open', 'CPRT'), ('Open', 'CRWD'), ('Open', 'CSCO'), ('Open', 'CSX'), ('Open', 'CTAS'), ('Open', 'DASH'), ('Open', 'DDOG'), ('Open', 'DX-Y.NYB'), ('Open', 'DXCM'), ('Open', 'EXC'), ('Open', 'FANG'), ('Open', 'FAST'), ('Open', 'FER'), ('Open', 'FTNT'), ('Open', 'GC=F'), ('Open', 'GEHC'), ('Open', 'GILD'), ('Open', 'GOOGL'), ('Open', 'HON'), ('Open', 'IDXX'), ('Open', 'INTC'), ('Open', 'INTU'), ('Open', 'ISRG'), ('Open', 'KDP'), ('Open', 'KLAC

In [ ]:
raw_time = raw_time.copy(True)

In [ ]:
raw_time.loc[:, (slice(None),'^NDX')]

Price,Open,Close,Volume
Ticker,^NDX,^NDX,^NDX
0,11059.169922,11229.730469,4415440000
1,11477.849609,11582.540039,5004780000
2,11429.469727,11573.179688,4091860000
3,11556.400391,11485.500000,4423320000
4,11272.459961,11039.469727,4634190000
...,...,...,...
797,25657.779297,25692.050781,8248280000
798,25758.820312,25627.949219,8533670000
799,25593.669922,25668.689453,7192600000


## NQ futures (overnight)

The Nasdaq keeps trading at night with the NQ futures (about 23h a day). We use 1-minute bars from Kaggle ("NQ Futures - 1min Bar 2022 2025", TGT Analytics, CC0), loaded at the top with `kagglehub` (`NQ`), so we don't need to download the CSV by hand. The timestamps are already in New York time, so we parse them as they are (no time-zone conversion).

**Anti-leakage rule:** one fixed cutoff, 09:00 ET of day `t`. We never use an NQ bar after 09:00 of day `t`, because during the NY session NQ and `^NDX` move almost the same. We don't use `Vwap_RTH` and `Vwap_ETH`.

In [ ]:
nq = NQ[["timestamp ET", "open", "high", "low", "close", "volume"]]
nq = nq.rename(columns={"timestamp ET": "timestamp"})
nq["timestamp"] = pd.to_datetime(nq["timestamp"], format="%m/%d/%Y %H:%M")
nq = nq.sort_values("timestamp").reset_index(drop=True)

In [ ]:
print(nq["timestamp"].min(), "->", nq["timestamp"].max(), len(nq))
print(nq.isnull().sum())

2022-12-26 18:01:00 -> 2025-12-11 20:52:00 1048575
timestamp    0
open         0
high         0
low          0
close        0
volume       0
dtype: int64


In [ ]:
# bars per calendar day (a full day is about 1380 bars)
bars_per_day = nq.groupby(nq["timestamp"].dt.normalize()).size()
print(bars_per_day.describe())

count     923.000000
mean     1136.050921
std       379.992020
min       285.000000
25%      1021.000000
50%      1380.000000
75%      1380.000000
max      1380.000000
dtype: float64


We observe that the total days is 923, instead of 1080, which is the supposed total length. We explain this beacuse of the saturdays were the market closes. Also, we observe that the mean is 1136, instead of 1380, however this is explain by the days that close early. Which is described by the percentiles.

In [ ]:
# bars per hour of the day: the night hours are there, and 17:00-18:00 is the daily break
print(nq.groupby(nq["timestamp"].dt.hour).size())

timestamp
0     45858
1     45889
2     45897
3     45896
4     45883
5     45900
6     45900
7     45900
8     45929
9     45885
10    45825
11    45835
12    45837
13    44288
14    44104
15    44100
16    44089
17      734
18    45171
19    45938
20    46007
21    45941
22    45886
23    45883
dtype: int64


In [ ]:
# the last day of the file is cut
print(bars_per_day.tail(3))

timestamp
2025-12-09    1380
2025-12-10    1380
2025-12-11    1193
dtype: int64


### Timestamp: start or end of the bar?

Globex opens at 18:00 and closes at 17:00. If the timestamp is the **end** of the minute, the first bar of a session is 18:01 and the last one is 17:00. If it is the **start**, the first bar is 18:00 and the last one is 16:59. So we count the bars stamped 17:00 and 18:00.

In [ ]:
print(nq.head(3))
hhmm = nq["timestamp"].dt.strftime("%H:%M")
print(hhmm[hhmm.isin(["16:59", "17:00", "18:00", "18:01"])].value_counts())

            timestamp      open      high      low     close  volume
0 2022-12-26 18:01:00  13759.00  13794.75  13759.0  13788.50     540
1 2022-12-26 18:02:00  13790.25  13794.00  13784.0  13790.25     304
2 2022-12-26 18:03:00  13789.75  13791.75  13782.5  13784.25     154
timestamp
18:01    764
16:59    735
17:00    734
Name: count, dtype: int64


In [ ]:
stamp_is_end = (hhmm == "17:00").sum() > (hhmm == "18:00").sum()
print("timestamp is the END of the bar:", stamp_is_end)

timestamp is the END of the bar: True


Result: the timestamp is the bar **end**. There are bars at 18:01 and 17:00 and no bar at 18:00. So the last bar we can use is the one stamped 09:00 (`timestamp <= 09:00`).

To avoid mistakes we build `bar_start` and `bar_end` for each bar and we always work with those two.

In [ ]:
one_min = pd.Timedelta(minutes=1)
nq["bar_end"] = nq["timestamp"] if stamp_is_end else nq["timestamp"] + one_min
nq["bar_start"] = nq["bar_end"] - one_min

### Sessions

We add 6 hours to the time, so 18:00 of the evening before becomes 00:00 of day `t`. In this shifted time the overnight window of day `t` is 00:00-15:00 (18:00 to 09:00 in NY), Asia is 00:00-09:00 and Europe is 09:00-15:00. Sunday evening goes to Monday.

In [ ]:
shifted = nq["bar_start"] + pd.Timedelta(hours=6)
nq["session_date"] = shifted.dt.normalize()
nq["h"] = shifted.dt.hour

In [ ]:
# overnight window, without the last (cut) day
last_day = nq["session_date"].max()
ovn = nq[(nq["h"] < 15) & (nq["session_date"] < last_day)].copy()
asia = ovn[ovn["h"] < 9]
europe = ovn[ovn["h"] >= 9]
print("last day dropped:", last_day.date())

last day dropped: 2025-12-12


In [ ]:
# anti-leakage: the latest bar used for day t ends at 09:00 of day t or before
last_used = ovn.groupby("session_date")["bar_end"].max()
assert (last_used <= last_used.index + pd.Timedelta(hours=9)).all()
print(last_used.dt.strftime("%H:%M").value_counts().head())

bar_end
09:00    766
Name: count, dtype: int64


In [ ]:
agg = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
o = ovn.groupby("session_date").agg(agg)
a = asia.groupby("session_date").agg(agg)
e = europe.groupby("session_date").agg(agg)

We keep only the days where `^NDX` trades (the futures also trade on some US holidays) and we drop the days with clearly too few overnight bars (a full night is 900 bars).

In [ ]:
n_bars = ovn.groupby("session_date").size()
few = n_bars[n_bars < 450].index
keep = o.index[o.index.isin(df_nasdaq.index) & ~o.index.isin(few)]
print("NQ sessions:", len(o), "| not a ^NDX day:", (~o.index.isin(df_nasdaq.index)).sum(),
      "| too few bars:", len(few), "| kept:", len(keep))

NQ sessions: 766 | not a ^NDX day: 23 | too few bars: 1 | kept: 742


In [ ]:
o = o.loc[keep]
a = a.reindex(keep)
e = e.reindex(keep)
# p09: NQ close of the last bar before the 09:00 cutoff
p09 = o["close"]

### Overnight features

All the features are returns, distances in % or ratios, never price levels, so they mean the same in 2023 and in 2025. We always compare NQ with NQ.

Returns of each session:

In [ ]:
nq_feat = pd.DataFrame(index=keep)
nq_feat["asia_ret"] = np.log(a["close"] / a["open"])
nq_feat["europe_ret"] = np.log(e["close"] / e["open"])
nq_feat["ovn_ret"] = np.log(p09 / o["open"])

Distance from the 09:00 price to the high and low of each session (support / resistance), and where the price is in the range of the night (0 = low, 1 = high).

In [ ]:
nq_feat["asia_high_dist"] = np.log(a["high"] / p09)
nq_feat["asia_low_dist"] = np.log(a["low"] / p09)
nq_feat["europe_high_dist"] = np.log(e["high"] / p09)
nq_feat["europe_low_dist"] = np.log(e["low"] / p09)
nq_feat["ovn_high_dist"] = np.log(o["high"] / p09)
nq_feat["ovn_low_dist"] = np.log(o["low"] / p09)
nq_feat["ovn_range_pos"] = (p09 - o["low"]) / (o["high"] - o["low"])

High and low of NQ during the NY session (09:30-16:00) of the previous trading day. This session is fully finished before the cutoff.

In [ ]:
minute = nq["bar_start"].dt.hour * 60 + nq["bar_start"].dt.minute
ny = nq[(minute >= 9 * 60 + 30) & (minute < 16 * 60)]
ny = ny.groupby(ny["bar_start"].dt.normalize()).agg({"high": "max", "low": "min", "close": "last"})
# previous trading day of ^NDX
prev_ny = ny.reindex(df_nasdaq.index).shift(1).reindex(keep)

In [ ]:
nq_feat["prev_ny_high_dist"] = np.log(prev_ny["high"] / p09)
nq_feat["prev_ny_low_dist"] = np.log(prev_ny["low"] / p09)

Volume. The "previous 20 days" means use `.shift(1).rolling(20)`, so day `t` is not in its own mean.

In [ ]:
ovn_volume = o["volume"]
nq_feat["ovn_volume_rel"] = ovn_volume / ovn_volume.shift(1).rolling(20).mean()
nq_feat["asia_volume_share"] = a["volume"] / ovn_volume
nq_feat["europe_volume_share"] = e["volume"] / ovn_volume

In [ ]:
# volume of the last hour before the cutoff (08:00-09:00), reaction to the 08:30 macro news
last_hour = ovn[ovn["h"] == 14].groupby("session_date")["volume"].sum().reindex(keep)
nq_feat["last_hour_volume_rel"] = last_hour / last_hour.shift(1).rolling(20).mean()

Volatility: std of the 1-minute log returns of the night, and the size of the night's range.

In [ ]:
ovn["lr"] = np.log(ovn["close"]).groupby(ovn["session_date"]).diff()
nq_feat["ovn_vol"] = ovn.groupby("session_date")["lr"].std().reindex(keep)
nq_feat["ovn_range"] = np.log(o["high"] / o["low"])

### Roll days

The contract changes every quarter (March, June, September, December). If the file is not adjusted, the price jumps between two sessions on the roll day. The overnight window is always inside one contract, so only `prev_ny_*_dist` would be affected (yesterday's contract against today's price).

First we look at the gap between the NQ close at 16:00 of the previous day and the first overnight open.

In [ ]:
roll_gap = np.log(o["open"] / prev_ny["close"])
print(roll_gap.abs().sort_values(ascending=False).head(20))

session_date
2025-04-03    0.036212
2025-04-07    0.024119
2025-04-14    0.021065
2025-02-03    0.017061
2025-05-27    0.013863
2025-04-23    0.013839
2023-05-25    0.013736
2025-05-12    0.012503
2024-04-26    0.012331
2024-04-25    0.010845
2025-10-31    0.010660
2024-08-29    0.009842
2025-11-20    0.009641
2025-05-01    0.009329
2025-07-31    0.009195
2025-06-20    0.008996
2025-05-19    0.008806
2025-04-16    0.008719
2025-06-23    0.008701
2024-01-31    0.008654
dtype: float64


The biggest gaps are news days (the tariff days of April 2025, earnings after the close...), not the quarterly roll dates. So this gap alone can't find the rolls.

A cleaner check: the difference in points between NQ and `^NDX` at the 16:00 close. A roll moves it by 150-250 points from one day to the next. We only use `^NDX` here for this check, not for a feature.

In [ ]:
spread = (ny["close"] - df_nasdaq[("Close", "^NDX")]).dropna()
print(spread.resample("QS").mean().round(0))
print(spread.diff().abs().sort_values(ascending=False).head(5))

bar_start
2022-10-01    2743.0
2023-01-01    2691.0
2023-04-01    2578.0
2023-07-01    2409.0
2023-10-01    2214.0
2024-01-01    2011.0
2024-04-01    1790.0
2024-07-01    1535.0
2024-10-01    1291.0
2025-01-01     992.0
2025-04-01     781.0
2025-07-01     586.0
2025-10-01     377.0
Freq: QS-JAN, dtype: float64
bar_start
2024-08-06    37.259766
2025-04-09    37.160156
2025-04-10    35.490234
2024-08-30    29.941406
2023-04-05    29.620117
dtype: float64


In [ ]:
ROLL_JUMP = 100
roll_days = spread.index[spread.diff().abs() > ROLL_JUMP]
print(len(roll_days), "roll days:", list(roll_days.date))

0 roll days: []


Result: there is no roll jump in this file. The difference goes down slowly (from about 2,700 points at the start of 2023 to about 300 at the end of 2025) and the biggest change in one day is under 40 points, so the series is already a back-adjusted continuous contract. No roll day to drop.

Limit to keep in mind: because of this adjustment the NQ prices of the first months are too high (about 25%), so the overnight returns of 2023 are a bit smaller than the real ones (about 20% smaller at the start, almost nothing at the end).

In [ ]:
nq_feat.describe().T

,count,mean,std,min,25%,50%,75%,max
asia_ret,742.0,0.000152,0.004065,-0.039395,-0.001696,0.000260,0.002006,0.016962
europe_ret,742.0,0.000168,0.004550,-0.020854,-0.001911,0.000301,0.002635,0.035733
ovn_ret,742.0,0.000323,0.005974,-0.044090,-0.002512,0.000582,0.003455,0.024257
asia_high_dist,742.0,0.002212,0.005245,-0.015072,-0.000877,0.001424,0.004322,0.044090
asia_low_dist,742.0,-0.003061,0.005366,-0.040411,-0.005783,-0.002681,-0.000147,0.013473
europe_high_dist,742.0,0.003035,0.003292,0.000000,0.000979,0.002085,0.003837,0.023730
europe_low_dist,742.0,-0.003497,0.003237,-0.039246,-0.004769,-0.002707,-0.001364,-0.000013
ovn_high_dist,742.0,0.003789,0.004231,0.000000,0.001174,0.002460,0.004775,0.044090
ovn_low_dist,742.0,-0.004683,0.004176,-0.040411,-0.006323,-0.003587,-0.001767,-0.000038
ovn_range_pos,742.0,0.557279,0.304068,0.003197,0.295547,0.596705,0.843776,1.000000


## Target

We decide at 09:00 ET of day `t`, we enter at the NY open of day `t` (09:30) and we exit at the NY close of the same day (16:00).

The target is the open-to-close log return of `^NDX`, and the label is 1 if it is positive, 0 if not. It is open-to-close on purpose: the overnight move is already in the open price. The `Open` and `Close` of `^NDX` of day `t` are only used here, never as features.

In [ ]:
ndx = raw_drop.set_index('Date')
close = ndx['Close']['^NDX']
open_ = ndx['Open']['^NDX']

In [ ]:
open_

Date
2022-10-03    11059.169922
2022-10-04    11477.849609
2022-10-05    11429.469727
2022-10-06    11556.400391
2022-10-07    11272.459961
                  ...     
2025-12-05    25657.779297
2025-12-08    25758.820312
2025-12-09    25593.669922
2025-12-10    25631.589844
2025-12-11    25598.390625
Name: ^NDX, Length: 802, dtype: float64

In [ ]:
ret_ny = np.log(close / open_)

In [ ]:
ret_ny

Date
2022-10-03    0.015305
2022-10-04    0.009080
2022-10-05    0.012495
2022-10-06   -0.006154
2022-10-07   -0.020886
                ...   
2025-12-05    0.001335
2025-12-08   -0.005094
2025-12-09    0.002927
2025-12-10    0.005635
2025-12-11    0.003443
Name: ^NDX, Length: 802, dtype: float64

In [ ]:
raw_time = raw_time.set_index("Date")

In [ ]:
raw_time

Price               Open         Close      Volume        Open              \
Ticker              ^NDX          ^NDX        ^NDX        AAPL        ABNB   
Date                                                                         
2022-10-03  11059.169922  11229.730469  4415440000  135.458539  105.489998   
2022-10-04  11477.849609  11582.540039  5004780000  142.142762  109.139999   
2022-10-05  11429.469727  11573.179688  4091860000  141.201857  110.519997   
2022-10-06  11556.400391  11485.500000  4423320000  142.907245  112.000000   
2022-10-07  11272.459961  11039.469727  4634190000  139.702324  108.870003   
...                  ...           ...         ...         ...         ...   
2025-12-05  25657.779297  25692.050781  8248280000  279.566033  120.919998   
2025-12-08  25758.820312  25627.949219  8533670000  277.164389  124.214996   
2025-12-09  25593.669922  25668.689453  7192600000  277.194290  121.610001   
2025-12-10  25631.589844  25776.439453  7987900000  276.785711  125.449997   
2025-12-11  25598.390625  25686.689453  8337770000  278.131020  127.599998   

Price                                                                   ...  \
Ticker            ADBE         ADI         ADP        ADSK         AEP  ...   
Date                                                                    ...   
2022-10-03  278.269989  132.337487  208.047447  187.830002   75.203027  ...   
2022-10-04  290.950012  138.866322  214.078860  197.669998   75.949027  ...   
2022-10-05  291.410004  139.662528  214.497956  196.139999   76.720787  ...   
2022-10-06  297.149994  141.751322  218.096766  203.809998   75.014360  ...   
2022-10-07  291.359985  138.323052  211.327371  201.589996   72.501886  ...   
...                ...         ...         ...         ...         ...  ...   
2025-12-05  330.600006  276.659838  252.083800  306.000000  114.353956  ...   
2025-12-08  347.450012  279.282235  253.502656  307.600006  114.033799  ...   
2025-12-09  341.399994  275.119470  251.782584  301.070007  112.811386  ...   
2025-12-10  342.130005  273.206590  250.101377  299.459991  113.180056  ...   
2025-12-11  341.649994  279.074086  252.268475  299.929993  111.094189  ...   

Price           Volume                                                       \
Ticker             WDC         WMT        XEL ^FVX ^IRX ^TNX ^TYX ^VIX ^VXN   
Date                                                                          
2022-10-03   5490185.0  17574900.0  2931500.0  0.0  0.0  0.0  0.0  0.0  0.0   
2022-10-04  13472374.0  16134300.0  2754400.0  0.0  0.0  0.0  0.0  0.0  0.0   
2022-10-05   9325827.0  14553900.0  3003000.0  0.0  0.0  0.0  0.0  0.0  0.0   
2022-10-06  11980823.0  14200500.0  3788300.0  0.0  0.0  0.0  0.0  0.0  0.0   
2022-10-07  12858766.0  17080500.0  6476400.0  0.0  0.0  0.0  0.0  0.0  0.0   
...                ...         ...        ...  ...  ...  ...  ...  ...  ...   
2025-12-05   5815300.0  23956400.0  3563000.0  0.0  0.0  0.0  0.0  0.0  0.0   
2025-12-08   4272200.0  22700700.0  3197700.0  0.0  0.0  0.0  0.0  0.0  0.0   
2025-12-09   3709900.0  24326700.0  4842400.0  0.0  0.0  0.0  0.0  0.0  0.0   
2025-12-10   7966000.0  22859600.0  4824700.0  0.0  0.0  0.0  0.0  0.0  0.0   
2025-12-11   7468900.0  21586600.0  5494900.0  0.0  0.0  0.0  0.0  0.0  0.0   

Price      Dayofweek  
Ticker                
Date                  
2022-10-03         0  
2022-10-04         1  
2022-10-05         2  
2022-10-06         3  
2022-10-07         4  
...              ...  
2025-12-05         4  
2025-12-08         0  
2025-12-09         1  
2025-12-10         2  
2025-12-11         3  

[802 rows x 322 columns]

In [ ]:
data = raw_time.copy()
# both have the Date index, so pandas aligns them by date
data["ret_ny"] = ret_ny

In [ ]:
r = data["ret_ny"]
data["y_ny"] = (r > 0).astype(float).where(r.notna())

In [ ]:
print(data[["ret_ny", "y_ny"]].tail(7))

Price         ret_ny y_ny
Ticker                   
Date                     
2025-12-03  0.006379  1.0
2025-12-04 -0.002984  0.0
2025-12-05  0.001335  1.0
2025-12-08 -0.005094  0.0
2025-12-09  0.002927  1.0
2025-12-10  0.005635  1.0
2025-12-11  0.003443  1.0


In [ ]:
print(data.columns.tolist())
print(data.index.min(), data.index.max(), len(data))

[('Open', '^NDX'), ('Close', '^NDX'), ('Volume', '^NDX'), ('Open', 'AAPL'), ('Open', 'ABNB'), ('Open', 'ADBE'), ('Open', 'ADI'), ('Open', 'ADP'), ('Open', 'ADSK'), ('Open', 'AEP'), ('Open', 'ALAB'), ('Open', 'ALNY'), ('Open', 'AMAT'), ('Open', 'AMD'), ('Open', 'AMGN'), ('Open', 'AMZN'), ('Open', 'APP'), ('Open', 'ARM'), ('Open', 'ASML'), ('Open', 'AVGO'), ('Open', 'AXON'), ('Open', 'BKNG'), ('Open', 'BKR'), ('Open', 'BTC-USD'), ('Open', 'CCEP'), ('Open', 'CDNS'), ('Open', 'CEG'), ('Open', 'CL=F'), ('Open', 'CMCSA'), ('Open', 'COST'), ('Open', 'CPRT'), ('Open', 'CRWD'), ('Open', 'CSCO'), ('Open', 'CSX'), ('Open', 'CTAS'), ('Open', 'DASH'), ('Open', 'DDOG'), ('Open', 'DX-Y.NYB'), ('Open', 'DXCM'), ('Open', 'EXC'), ('Open', 'FANG'), ('Open', 'FAST'), ('Open', 'FER'), ('Open', 'FTNT'), ('Open', 'GC=F'), ('Open', 'GEHC'), ('Open', 'GILD'), ('Open', 'GOOGL'), ('Open', 'HON'), ('Open', 'IDXX'), ('Open', 'INTC'), ('Open', 'INTU'), ('Open', 'ISRG'), ('Open', 'KDP'), ('Open', 'KLAC'), ('Open', '

The target doesn't look ahead anymore, so the last rows keep their target. We only remove the rows without target (if any).

In [ ]:
target_cols = [("ret_ny", ""), ("y_ny", "")]

data = data.dropna(subset=[("y_ny", "")])

## Daily features

Context from the day before. All the daily US data closes at 16:00 or later, so everything here is shifted by 1 day (`shift(1)`): for day `t` we only use values up to the close of `t-1`.

Nasdaq-100: momentum over 1, 5 and 20 days, the NY session return of the day before, and the 20-day volatility of the daily returns.

In [ ]:
ndx_close = data[("Close", "^NDX")]
for n in (1, 5, 20):
    data[(f"ndx_ret_{n}d", "")] = np.log(ndx_close / ndx_close.shift(n)).shift(1)

In [ ]:
data[("ndx_ret_ny_prev", "")] = data[("ret_ny", "")].shift(1)
data[("ndx_vol_20d", "")] = np.log(ndx_close).diff().rolling(20).std().shift(1)

Macro: level for the volatility indices and the yields, daily log return for the dollar, oil and gold. Some of them don't trade on a few Nasdaq days (bond holidays), so we forward fill first (it only uses the past). Silver and the 5y / 30y yields are not used.

In [ ]:
macro_close = data["Close"][["^VIX", "^VXN", "^TNX", "^IRX", "DX-Y.NYB", "CL=F", "GC=F"]].ffill()

In [ ]:
data[("vix", "")] = macro_close["^VIX"].shift(1)
data[("vxn", "")] = macro_close["^VXN"].shift(1)
data[("yield_10y", "")] = macro_close["^TNX"].shift(1)
data[("yield_3m", "")] = macro_close["^IRX"].shift(1)

In [ ]:
data[("dollar_ret", "")] = np.log(macro_close["DX-Y.NYB"]).diff().shift(1)
data[("oil_ret", "")] = np.log(macro_close["CL=F"]).diff().shift(1)
data[("gold_ret", "")] = np.log(macro_close["GC=F"]).diff().shift(1)

Bitcoin (risk sentiment). A yfinance `BTC-USD` daily bar dated `D` covers 00:00-24:00 UTC, so it closes around 19:00-20:00 ET of day `D`: the bar dated `t-1` is known before the 09:00 cutoff of day `t`, the bar dated `t` is not.

`btc_ret` is the BTC return since the last trading day (Monday's value covers the whole weekend). `btc_vol_7d` is the std of the BTC daily log returns over the 7 calendar days ending at `t-1`.

In [ ]:
days = pd.Series(data.index, index=data.index)
btc_last = days - pd.Timedelta(days=1)             # calendar day t-1
btc_first = days.shift(1) - pd.Timedelta(days=1)   # calendar day before the previous trading day
# the BTC values of day t only use bars dated t-1 or before
assert (btc_last < days).all() and (btc_first.dropna() < btc_last[btc_first.notna()]).all()

In [ ]:
data[("btc_ret", "")] = np.log(btc_close.reindex(btc_last).values / btc_close.reindex(btc_first).values)
btc_vol = np.log(btc_close).diff().rolling(7).std()
data[("btc_vol_7d", "")] = btc_vol.reindex(btc_last).values

Stocks: daily log return of each stock, shifted by 1. We keep only the stocks with no missing values in the period. They are kept for a PCA that we will do later, fitted on train only.

In [ ]:
not_stocks = tickers1 + tickers2 + ["^NDX"]
stocks = [t for t in data["Close"].columns if t not in not_stocks]
stock_close = data["Close"][stocks].dropna(axis=1)
print(len(stocks), "stocks,", stock_close.shape[1], "with no missing values")

95 stocks, 92 with no missing values


In [ ]:
stock_ret = np.log(stock_close).diff().shift(1)
stock_ret.columns = pd.MultiIndex.from_product([["stock_ret"], stock_ret.columns])

Breadth: % of those stocks with a positive return the day before.

In [ ]:
data[("pct_up", "")] = ((stock_ret > 0).mean(axis=1) * 100).where(stock_ret.notna().all(axis=1))

## Join

The Open, Close and Volume levels are not features (they are replaced by the returns above), so we drop them. Then we do an inner join of the daily table and the NQ features on the `^NDX` trading dates.

In [ ]:
data = data.drop(columns=["Open", "Close", "Volume"], level=0)
data = pd.concat([data, stock_ret], axis=1)

In [ ]:
nq_feat.columns = pd.MultiIndex.from_product([nq_feat.columns, [""]])
nq_cols = nq_feat.columns.tolist()
print("daily rows:", len(data), "| NQ rows:", len(nq_feat))
data = data.join(nq_feat, how="inner")
print("after the join:", len(data))

daily rows: 802 | NQ rows: 742
after the join: 742


In [ ]:
# drop the roll days
n_roll = data.index.isin(roll_days).sum()
data = data[~data.index.isin(roll_days)]
print("roll days dropped:", n_roll)

roll days dropped: 0


The first rows have no 20-day means (warm-up of the NQ file), so they have missing values. We remove the rows with missing values.

In [ ]:
nulls = data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

last_hour_volume_rel      20
ovn_volume_rel            20
prev_ny_low_dist           1
prev_ny_high_dist          1
dtype: int64


In [ ]:
n_before = len(data)
data = data.dropna()
print("rows dropped:", n_before - len(data), "| first day:", data.index.min().date())

rows dropped: 20 | first day: 2023-01-26


In [ ]:
# anti-leakage: no price level of day t in the features
feature_cols = [c for c in data.columns if c not in target_cols]
assert ("Close", "^NDX") not in feature_cols and ("Open", "^NDX") not in feature_cols
assert not data.columns.get_level_values(0).isin(["Open", "Close", "Volume"]).any()

In [ ]:
print(data.shape)
print(data.index.min(), data.index.max())

(722, 128)
2023-01-26 00:00:00 2025-12-11 00:00:00


In [ ]:
data.to_feather("data/final_data_ny.feather")

## Train / test split

The split is chronological: the test set is the last 20% of the rows and we don't touch it until the end. The test set starts on 2025-05-16 (144 rows, train has 578).

The target doesn't look ahead anymore (it ends at the close of day `t`), so we don't need to remove rows of train. We keep `gap = 1` in `TimeSeriesSplit`.

In [ ]:
n_test = int(len(data) * 0.2)
TEST_START = data.index[-n_test]
train = data.loc[data.index < TEST_START]
test = data.loc[TEST_START:]

In [ ]:
gap = 1

In [ ]:
print("train:", train.index.min(), "->", train.index.max(), len(train))
print("test: ", test.index.min(), "->", test.index.max(), len(test))
print("TEST_START:", TEST_START.date())

train: 2023-01-26 00:00:00 -> 2025-05-15 00:00:00 578
test:  2025-05-16 00:00:00 -> 2025-12-11 00:00:00 144
TEST_START: 2025-05-16


In [ ]:
test.to_feather("data/test_holdout_ny.feather")

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, gap=gap)
X_train = train.drop(columns=target_cols)
y_train = train[[("y_ny", "")]]

Class balance: in train the NY session goes up 55.5% of the days (56.9% in test). So a model that always says "up" already gets that accuracy, that's the baseline to beat.

In [ ]:
y_train.describe()

,y_ny
,
count,578.000000
mean,0.555363
std,0.497356
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


**Known limitation:** the table has one row per day and the NQ file only covers Dec 2022 - Dec 2025, so we only have 722 rows (578 for train and 144 for test). It is small. Possible options for later: a longer NQ source (since 2008, not on Kaggle), or a panel with one row per stock per day.

## Checks

In [ ]:
print("final shape:", data.shape)
print("train:", train.shape, train.index.min().date(), "->", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "->", test.index.max().date())
print("roll days dropped:", n_roll)

final shape: (722, 128)
train: (578, 128) 2023-01-26 -> 2025-05-15
test:  (144, 128) 2025-05-16 -> 2025-12-11
roll days dropped: 0


In [ ]:
print([c for c in feature_cols if c[0] != "stock_ret"])
print((data.columns.get_level_values(0) == "stock_ret").sum(), "stock_ret columns")

[('Dayofweek', ''), ('ndx_ret_1d', ''), ('ndx_ret_5d', ''), ('ndx_ret_20d', ''), ('ndx_ret_ny_prev', ''), ('ndx_vol_20d', ''), ('vix', ''), ('vxn', ''), ('yield_10y', ''), ('yield_3m', ''), ('dollar_ret', ''), ('oil_ret', ''), ('gold_ret', ''), ('btc_ret', ''), ('btc_vol_7d', ''), ('pct_up', ''), ('asia_ret', ''), ('europe_ret', ''), ('ovn_ret', ''), ('asia_high_dist', ''), ('asia_low_dist', ''), ('europe_high_dist', ''), ('europe_low_dist', ''), ('ovn_high_dist', ''), ('ovn_low_dist', ''), ('ovn_range_pos', ''), ('prev_ny_high_dist', ''), ('prev_ny_low_dist', ''), ('ovn_volume_rel', ''), ('asia_volume_share', ''), ('europe_volume_share', ''), ('last_hour_volume_rel', ''), ('ovn_vol', ''), ('ovn_range', '')]
92 stock_ret columns


In [ ]:
print(train[("y_ny", "")].value_counts(normalize=True))
print(test[("y_ny", "")].value_counts(normalize=True))

(y_ny, )
1.0    0.555363
0.0    0.444637
Name: proportion, dtype: float64
(y_ny, )
1.0    0.569444
0.0    0.430556
Name: proportion, dtype: float64


Leakage sanity check: correlation of each feature with the target return on train (NQ features first, then the daily features). A value above 0.3 (in absolute value) would be suspicious.

In [ ]:
corr = train[nq_cols].corrwith(train[("ret_ny", "")])
corr.index = [c[0] for c in corr.index]
print(corr.sort_values())
print("max |corr|:", corr.abs().max())

asia_ret               -0.116855
europe_low_dist        -0.081548
asia_low_dist          -0.063654
ovn_low_dist           -0.063242
ovn_ret                -0.058614
asia_volume_share      -0.054364
ovn_range_pos          -0.028815
europe_ret              0.025273
ovn_volume_rel          0.042284
asia_high_dist          0.043686
europe_volume_share     0.054364
last_hour_volume_rel    0.060476
prev_ny_low_dist        0.081445
ovn_high_dist           0.097493
ovn_range               0.122001
europe_high_dist        0.126657
prev_ny_high_dist       0.157110
ovn_vol                 0.249443
dtype: float64
max |corr|: 0.24944336881790719


In [ ]:
daily_cols = [c for c in feature_cols if c[0] != "stock_ret" and c not in nq_cols]
corr_daily = train[daily_cols].corrwith(train[("ret_ny", "")])
corr_daily.index = [c[0] for c in corr_daily.index]
print(corr_daily.sort_values())
print("max |corr|:", corr_daily.abs().max())

ndx_ret_ny_prev   -0.162624
btc_ret           -0.069112
ndx_ret_1d        -0.063381
ndx_ret_5d        -0.060737
Dayofweek         -0.057932
yield_10y         -0.052061
ndx_ret_20d       -0.043620
pct_up            -0.036114
oil_ret           -0.026860
yield_3m          -0.025598
dollar_ret        -0.009711
gold_ret          -0.009263
btc_vol_7d         0.014104
ndx_vol_20d        0.069231
vxn                0.095714
vix                0.142906
dtype: float64
max |corr|: 0.16262354262429152


Next: EDA on train.